# 🧩 Mini-Lab: NumPy Where It Actually Matters: Shape and dtype

**Module: Real Data: Paths, CSV, JSON, pandas, NumPy** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** how NumPy silently broadcasts arrays of mismatched shapes, producing wrong results without errors
2. **Assert** array shape and dtype before computation to catch bugs immediately
3. **Recognize** common shape/dtype pitfalls in AI-engineering data (e.g., embeddings, token arrays)

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Array Shape & dtype | Understand (d,) versus (1, d) and data types (dtype), then assert expected shapes so NumPy does not silently broadcast and change your math. |


## Why This Matters

NumPy is designed to be **forgiving** — if two arrays don't have matching shapes, it will try to *broadcast* one to fit the other. This is powerful when intentional, but **dangerous** when accidental:

- You subtract a `(3,)` vector from a `(3, 4)` matrix and get a result — but is it the result you wanted?
- You pass `float32` embeddings into a function expecting `float64` and lose precision — silently.
- You load 100 embeddings but your labels array has 99 entries — NumPy won't stop you from indexing until it's too late.

**One `assert` before the math catches the bug instantly.** Without it, you get wrong numbers and no error.

In [1]:
import numpy as np

## Silent Broadcasting: A Bug You Won't Notice

Let's say you have 3 embeddings (each of dimension 4) and you want to subtract a **single** mean vector. Broadcasting handles this correctly. But what if you accidentally pass in the wrong shape?

In [2]:
# 3 embeddings, each 4-dimensional
embeddings = np.array([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10., 11., 12.],
])
print(f"embeddings shape: {embeddings.shape}")  # (3, 4)

# Correct mean vector: shape (4,)
mean_vec = embeddings.mean(axis=0)
print(f"mean_vec shape:   {mean_vec.shape}")    # (4,)

# This works correctly — broadcasting subtracts (4,) from each row of (3, 4)
centered = embeddings - mean_vec
print(f"\nCentered (correct):\n{centered}")

embeddings shape: (3, 4)
mean_vec shape:   (4,)

Centered (correct):
[[-4. -4. -4. -4.]
 [ 0.  0.  0.  0.]
 [ 4.  4.  4.  4.]]


In [3]:
# Now the BUG: accidentally compute mean along the WRONG axis
wrong_mean = embeddings.mean(axis=1)  # shape (3,) — one value per row, not per column
print(f"wrong_mean shape: {wrong_mean.shape}")  # (3,)

# NumPy does NOT raise an error — it broadcasts (3,) as a column across (3, 4)
wrong_result = embeddings - wrong_mean.reshape(-1, 1)  # runs fine, wrong answer
print(f"\nWrong result (no error raised!):\n{wrong_result}")
print(f"\n⚠️  The shapes were compatible, so NumPy happily computed garbage.")

wrong_mean shape: (3,)

Wrong result (no error raised!):
[[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]

⚠️  The shapes were compatible, so NumPy happily computed garbage.


## The Fix: Assert Before You Compute

A single `assert` statement catches the bug **immediately** — before any math runs.

In [4]:
def center_embeddings(emb: np.ndarray, mean: np.ndarray) -> np.ndarray:
    """Subtract a mean vector from each embedding row."""
    assert emb.ndim == 2, f"Expected 2D embeddings, got {emb.ndim}D"
    assert mean.shape == (emb.shape[1],), (
        f"Mean shape {mean.shape} doesn't match embedding dim ({emb.shape[1]},)"
    )
    return emb - mean


# ✅ Correct usage — passes assertion
result = center_embeddings(embeddings, mean_vec)
print(f"✅ Correct centering passed. Result shape: {result.shape}")

✅ Correct centering passed. Result shape: (3, 4)


In [5]:
# ❌ Wrong usage — assertion catches it instantly
try:
    center_embeddings(embeddings, wrong_mean)
except AssertionError as e:
    print(f"❌ Caught the bug: {e}")

❌ Caught the bug: Mean shape (3,) doesn't match embedding dim (4,)


## Asserting dtype: Catch Type Mismatches

Shape isn't the only thing that can go wrong. A common AI-engineering pitfall: your embeddings are `float32` (from the API) but your math assumes `float64`, or worse, integers sneak in.

In [6]:
# Simulate embeddings returned as float32 (common from APIs and model outputs)
emb_f32 = np.array([[0.1, 0.2], [0.3, 0.4]], dtype=np.float32)

# Simulate accidentally loading token counts as embeddings
not_embeddings = np.array([[1, 2], [3, 4]])  # int64!

print(f"emb_f32 dtype:       {emb_f32.dtype}")
print(f"not_embeddings dtype: {not_embeddings.dtype}")

emb_f32 dtype:       float32
not_embeddings dtype: int64


In [7]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors with safety checks."""
    assert a.shape == b.shape, f"Shape mismatch: {a.shape} vs {b.shape}"
    assert np.issubdtype(a.dtype, np.floating), f"Expected float array, got {a.dtype}"
    assert np.issubdtype(b.dtype, np.floating), f"Expected float array, got {b.dtype}"
    
    dot = np.dot(a, b)
    return dot / (np.linalg.norm(a) * np.linalg.norm(b))


# ✅ Works — both are float
sim = cosine_similarity(emb_f32[0], emb_f32[1])
print(f"✅ Cosine similarity: {sim:.4f}")

# ❌ Caught — integers passed in
try:
    cosine_similarity(not_embeddings[0], not_embeddings[1])
except AssertionError as e:
    print(f"❌ Caught dtype bug: {e}")

✅ Cosine similarity: 0.9839
❌ Caught dtype bug: Expected float array, got int64


## Mismatched Lengths: Labels vs Data

Another classic silent bug: your data array has a different number of rows than your labels array. NumPy won't stop you from zipping or indexing them — you'll just get wrong pairings.

In [8]:
# 4 data points but only 3 labels — a common off-by-one bug
data = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6], [0.7, 0.8]])
labels = np.array(["pos", "neg", "pos"])  # oops — missing one label

print(f"data rows:   {data.shape[0]}")
print(f"label count: {labels.shape[0]}")

# Without assertion, you'd silently pair wrong labels to wrong data
# The assertion catches it:
try:
    assert data.shape[0] == labels.shape[0], (
        f"Row count mismatch: {data.shape[0]} data rows vs {labels.shape[0]} labels"
    )
except AssertionError as e:
    print(f"\n❌ Caught: {e}")

data rows:   4
label count: 3

❌ Caught: Row count mismatch: 4 data rows vs 3 labels


## The Assertion Checklist

Before any array computation, check the three things that cause silent bugs:

```python
# 1. Dimensions — is it 1D, 2D, etc.?
assert arr.ndim == 2, f"Expected 2D, got {arr.ndim}D"

# 2. Shape — do the sizes match expectations?
assert arr.shape == (n_rows, n_cols), f"Unexpected shape: {arr.shape}"

# 3. dtype — is it the right number type?
assert np.issubdtype(arr.dtype, np.floating), f"Expected float, got {arr.dtype}"
```

These three lines cost nothing at runtime and save hours of debugging.

## 🎯 Summary

### Key Takeaways

1. **Silent broadcasting is the danger** — NumPy won't error on many shape mismatches; it will broadcast and give you a wrong answer with no warning
2. **Assert shape before computing** — `assert arr.shape == (expected,)` catches mismatched dimensions, wrong axes, and off-by-one row counts instantly
3. **Assert dtype too** — passing integers where floats are expected (or vice versa) produces subtle precision bugs; `np.issubdtype` catches them
4. **Three checks, zero cost** — asserting `ndim`, `shape`, and `dtype` at the top of a function is a one-time habit that prevents the hardest-to-find numerical bugs